# Day 23 – Feature Engineering
## Turning Raw Data into Useful Predictors

Feature engineering transforms raw fields into variables that a model can use more effectively.

**Framework:** Raw Data → Transform → Represent → Validate → Model


## 1. Create a Synthetic Public-Grievance Dataset
We will model average resolution time using complaint volume, backlog, staffing and priority indicators.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
n = 120

df = pd.DataFrame({
    "month": pd.date_range("2016-01-01", periods=n, freq="MS"),
    "complaint_volume": np.random.randint(800, 5000, n),
    "backlog": np.random.randint(50, 900, n),
    "weekend_share": np.random.uniform(0.08, 0.30, n),
    "priority_share": np.random.uniform(0.08, 0.35, n),
    "staff_available": np.random.randint(35, 90, n)
})

df["avg_resolution_days"] = (
    2.5
    + 0.0015 * df["complaint_volume"]
    + 0.004 * df["backlog"]
    + 8 * df["priority_share"]
    - 0.035 * df["staff_available"]
    + np.random.normal(0, 1.5, n)
).clip(lower=1)

df.head()


## 2. Ratio Features
Ratios can express operational pressure better than absolute counts.

In [ ]:
df["complaints_per_staff"] = df["complaint_volume"] / df["staff_available"]
df["backlog_per_staff"] = df["backlog"] / df["staff_available"]
df["priority_count_est"] = df["complaint_volume"] * df["priority_share"]
df["backlog_ratio"] = df["backlog"] / df["complaint_volume"]
df["priority_to_staff"] = df["priority_count_est"] / df["staff_available"]

df[["complaints_per_staff","backlog_per_staff","backlog_ratio","priority_to_staff"]].head()


## 3. Date and Time Features
Calendar variables can capture seasonality without treating dates as plain text.

In [ ]:
df["year"] = df["month"].dt.year
df["month_number"] = df["month"].dt.month
df["quarter"] = df["month"].dt.quarter
df["month_sin"] = np.sin(2 * np.pi * df["month_number"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month_number"] / 12)

df[["month","year","month_number","quarter","month_sin","month_cos"]].head(14)


## 4. Lag and Rolling Features
For time-dependent operational data, previous-period and recent-average values can be useful predictors.

In [ ]:
df = df.sort_values("month").reset_index(drop=True)

df["volume_lag_1"] = df["complaint_volume"].shift(1)
df["backlog_lag_1"] = df["backlog"].shift(1)
df["resolution_lag_1"] = df["avg_resolution_days"].shift(1)
df["volume_change"] = df["complaint_volume"].pct_change()

df["volume_rolling_3"] = df["complaint_volume"].rolling(3).mean()
df["backlog_rolling_3"] = df["backlog"].rolling(3).mean()
df["resolution_rolling_3"] = df["avg_resolution_days"].rolling(3).mean()

df[["month","complaint_volume","volume_lag_1","volume_rolling_3"]].tail(10)


## 5. Categorical Encoding
Suppose records belong to operational zones. One-hot encoding converts categories into model-ready indicators.

In [ ]:
zones = ["North", "South", "Central", "East"]
df["zone"] = np.random.choice(zones, len(df))
zone_encoded = pd.get_dummies(df["zone"], prefix="zone", dtype=int)
zone_encoded.head()


## 6. Feature Scaling
Standardization puts numeric variables on comparable scales and is useful for many distance-based and regularized algorithms.

In [ ]:
from sklearn.preprocessing import StandardScaler

scale_cols = ["complaint_volume","backlog","staff_available","complaints_per_staff"]
scaler = StandardScaler()
scaled = pd.DataFrame(
    scaler.fit_transform(df[scale_cols]),
    columns=scale_cols
)
scaled.head()


## 7. Leakage Check
A feature must be available at the time the prediction is made. Using future information or information derived from the target creates **data leakage** and can make validation misleading.


In [ ]:
feature_cols = [
    "complaint_volume", "backlog", "weekend_share", "priority_share",
    "staff_available", "complaints_per_staff", "backlog_per_staff",
    "priority_count_est", "backlog_ratio", "priority_to_staff",
    "month_sin", "month_cos", "volume_lag_1", "backlog_lag_1",
    "volume_rolling_3", "backlog_rolling_3"
]

model_df = df[feature_cols + ["avg_resolution_days"]].dropna()
X = model_df[feature_cols]
y = model_df["avg_resolution_days"]
X.shape, y.shape


## 8. Compare a Baseline Model with an Engineered Model
Because this is time-oriented data, use a chronological train/test split rather than randomly mixing future observations into training.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

split = int(len(model_df) * 0.80)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

baseline_cols = ["complaint_volume", "backlog", "staff_available"]

baseline = LinearRegression().fit(X_train[baseline_cols], y_train)
engineered = LinearRegression().fit(X_train, y_train)

def evaluate(model, Xte, yte):
    pred = model.predict(Xte)
    return {
        "MAE": mean_absolute_error(yte, pred),
        "RMSE": mean_squared_error(yte, pred) ** 0.5,
        "R2": r2_score(yte, pred)
    }

results = pd.DataFrame({
    "Baseline": evaluate(baseline, X_test[baseline_cols], y_test),
    "Engineered": evaluate(engineered, X_test, y_test)
}).T

results


## 9. Inspect Feature Effects
For a linear model, coefficients provide one way to inspect the direction and magnitude of modeled relationships. Interpretation must consider scale and correlation among features.

In [ ]:
coef = pd.Series(engineered.coef_, index=X.columns)
coef.reindex(coef.abs().sort_values(ascending=False).index).head(10)


## 10. Visualize an Engineered Feature

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["backlog_per_staff"], df["avg_resolution_days"], alpha=0.7)
plt.xlabel("Backlog per Staff")
plt.ylabel("Average Resolution Days")
plt.title("Engineered Feature vs Target")
plt.tight_layout()
plt.show()


## 11. Feature Engineering Checklist

- Define the prediction objective first.
- Understand when each feature becomes available.
- Create ratios when operational pressure matters more than volume alone.
- Extract useful information from dates.
- Use lags and rolling statistics for time-dependent problems.
- Encode categorical variables appropriately.
- Scale features when the algorithm benefits from comparable magnitudes.
- Check carefully for data leakage.
- Validate engineered features on unseen data.
- Prefer features that improve both predictive performance and operational interpretability.

## Key Takeaway

**Feature engineering is the bridge between raw data and model-ready information.**

A simple model with well-designed, leakage-free features can be more useful than a complex model built on poorly represented data.

**Next:** Day 24 – Feature Selection & Dimensionality Reduction.
